In [ ]:
!pip install -r requirements.txt

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

driver = webdriver.Chrome()
wait = WebDriverWait(driver, 300)

driver.get("https://www.imdb.com/search/title/?year=2024")

movies = []
collected_count = 0

while True:

    wait.until(EC.presence_of_all_elements_located(
        (By.CLASS_NAME, "ipc-metadata-list-summary-item")
    ))

    movie_cards = driver.find_elements(
        By.CLASS_NAME, "ipc-metadata-list-summary-item"
    )

    total_on_page = len(movie_cards)
    print(f"Currently loaded: {total_on_page}")

    # Collect only newly added cards
    for card in movie_cards[collected_count:]:
        try:
            title = card.find_element(By.CLASS_NAME, "ipc-title__text").text
        except:
            title = "Not Available"

        try:
            storyline = card.find_element(By.CLASS_NAME, "ipc-html-content-inner-div").text
        except:
            storyline = "Not Available"

        movies.append([title, storyline])

    collected_count = total_on_page

    # Try clicking 50 more
    try:
        button = driver.find_element(
            By.XPATH,
            "//button[.//span[contains(text(),'50 more')]]"
        )

        driver.execute_script("arguments[0].scrollIntoView();", button)
        time.sleep(2)

        driver.execute_script("arguments[0].click();", button)

        # Wait until new cards are added
        wait.until(lambda d: len(
            d.find_elements(By.CLASS_NAME, "ipc-metadata-list-summary-item")
        ) > total_on_page)

        time.sleep(2)

    except:
        print("No more 50 more button.")
        break

df = pd.DataFrame(movies, columns=["Movie Name", "Storyline"])
df.to_csv("imdb_2024_movies.csv", index=False)
df.to_csv("imdb_2024_movies.txt", index=False, sep="\t")

driver.quit()

print(f"Scraping completed. Total movies collected: {len(movies)}")

Currently loaded: 50
Currently loaded: 100
Currently loaded: 150
Currently loaded: 200
Currently loaded: 250
Currently loaded: 300
Currently loaded: 350
Currently loaded: 400
Currently loaded: 450
Currently loaded: 500
Currently loaded: 550
Currently loaded: 600
Currently loaded: 650
Currently loaded: 700
Currently loaded: 750
Currently loaded: 800
Currently loaded: 850
Currently loaded: 900
Currently loaded: 950
Currently loaded: 1000
Currently loaded: 1050
Currently loaded: 1100
Currently loaded: 1150
Currently loaded: 1200
Currently loaded: 1250
Currently loaded: 1300
Currently loaded: 1350
Currently loaded: 1400
Currently loaded: 1450
Currently loaded: 1500
Currently loaded: 1550
Currently loaded: 1600
Currently loaded: 1650
Currently loaded: 1700
Currently loaded: 1750
Currently loaded: 1800
Currently loaded: 1850
Currently loaded: 1900
Currently loaded: 1950
Currently loaded: 2000
Currently loaded: 2050
Currently loaded: 2100
Currently loaded: 2150
Currently loaded: 2200
Currentl

### Import required packages

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

In [2]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ragav\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ragav\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
stop_words = set(stopwords.words('english'))

### Load the scraped data

In [5]:
df = pd.read_csv("imdb_2024_movies.txt", sep="\t")

In [6]:
df

,Movie Name,Storyline
0,1. High Potential,A single mom with three kids helps solve an un...
1,2. Landman,A modern-day tale of fortune seeking in the wo...
2,3. Fallout,"In a future, post-apocalyptic Los Angeles brou..."
3,4. Tracker,Colter Shaw travels the country in his old-sch...
4,5. Ted,"It's 1993, and Ted the bear's moment of fame h..."
...,...,...
9495,9496. The War Games in Colour,The Doctor and friends arrive on an unnamed pl...
9496,9497. Skydance's Behemoth,Skydance's BEHEMOTH is a sprawling original st...
9497,9498. Shadowland,Leave your troubled past behind and get a fres...
9498,9499. Samadhana Pusthakam,Set against the backdrop of a school and the s...


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9500 entries, 0 to 9499
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Movie Name  9500 non-null   str  
 1   Storyline   9500 non-null   str  
dtypes: str(2)
memory usage: 2.0 MB


In [8]:
df.isnull().value_counts()

Movie Name  Storyline
False       False        9500
Name: count, dtype: int64

In [9]:
df.describe()

,Movie Name,Storyline
count,9500,9500
unique,9500,8846
top,1. High Potential,Not Available
freq,1,653


In [10]:
df = df.replace("Not Available", np.nan)

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9500 entries, 0 to 9499
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Movie Name  9500 non-null   str  
 1   Storyline   8847 non-null   str  
dtypes: str(2)
memory usage: 2.0 MB


In [12]:
df = df.dropna()

In [13]:
df.info()

<class 'pandas.DataFrame'>
Index: 8847 entries, 0 to 9499
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Movie Name  8847 non-null   str  
 1   Storyline   8847 non-null   str  
dtypes: str(2)
memory usage: 2.0 MB


### Clean the text

In [14]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    return " ".join(tokens)


In [15]:
df["cleaned_storyline"] = df["Storyline"].apply(clean_text)
df.to_csv("cleaned_data.csv", index=False)

In [16]:
df.head()

,Movie Name,Storyline,cleaned_storyline
0,1. High Potential,A single mom with three kids helps solve an un...,single mom three kids helps solve unsolvable c...
1,2. Landman,A modern-day tale of fortune seeking in the wo...,modern day tale fortune seeking world west tex...
2,3. Fallout,"In a future, post-apocalyptic Los Angeles brou...",future post apocalyptic los angeles brought nu...
3,4. Tracker,Colter Shaw travels the country in his old-sch...,colter shaw travels country old school rv help...
4,5. Ted,"It's 1993, and Ted the bear's moment of fame h...",ted bear moment fame passed living back home b...


### Text Vectorization

In [17]:
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
tfidf_matrix = vectorizer.fit_transform(df["cleaned_storyline"])

### Cosine Similarity

In [18]:
cosine_sim = cosine_similarity(tfidf_matrix)

In [19]:
def recommend_movies(user_input, df, vectorizer, tfidf_matrix):
    cleaned_input = clean_text(user_input)
    input_vector = vectorizer.transform([cleaned_input])
    
    similarity_scores = cosine_similarity(input_vector, tfidf_matrix)
    
    top_indices = similarity_scores.argsort()[0][-5:][::-1]
    
    return df.iloc[top_indices][["Movie Name", "Storyline"]]

### Test the model 

In [20]:
user_input = "A modern-day tale of fortune seeking in the world of West Texas oil rigs"
result = recommend_movies(user_input, df, vectorizer, tfidf_matrix)
result

,Movie Name,Storyline
1,2. Landman,A modern-day tale of fortune seeking in the wo...
1849,1850. Very Parivarik,A modern-day couple who manages to adjust to t...
4428,4429. Death Streamer,A modern-day vampire employs technologically a...
8528,8529. A Salad Bowl of Eccentrics,A jaded private eye's world turns upside down ...
7534,7535. Danny Dyer: How to Be a Man,With traditional gender roles a thing of the p...
